# W3D4_LAB_Slice_docs_the_way_Google_wishes_you_would
- Week 3 / Day 4
- Student: Andreas Papachristophorou
- Course: AI Consulting & Integration 2026-07
- Date: 2026-07-23

## Step 1: Setup and Data Loading

In [2]:
# Libraries Imports 

import os
import json
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydub import AudioSegment
from tempfile import TemporaryDirectory

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set")

# Create the OpenAI client so we can call Whisper.
client = OpenAI(api_key=api_key)

print(f"\nOPENAI Initiated")


OPENAI Initiated


**1.1** Load the podcast audio file and Transcribe the audio file using Whisper.

In [3]:
import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydub import AudioSegment
from tempfile import TemporaryDirectory

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set")

client = OpenAI(api_key=api_key)

# 1. Point and load the audio file
audio_dir = Path("sources")
audio_filename = "The_Blueprint_For_Trustworthy_AI.m4a"
audio_path = audio_dir / audio_filename

if audio_path.exists():
    print("The audio file is ready.")
    print("Audio path:", audio_path)
else:
    print("The audio file was not found. Put it inside the 'sources' folder.")

# 2. Function that sends the audio file with extra context (prompt)
def transcribe_with_prompt(audio_path: Path, prompt_text: str, max_chunk_mb: int = 24) -> str:
    """Transcribe audio, splitting it into chunks if the file is too large."""
    audio = AudioSegment.from_file(audio_path)
    max_chunk_bytes = max_chunk_mb * 1024 * 1024
    audio_file_size = audio_path.stat().st_size

    def transcribe_file(file_path: Path) -> str:
        with open(file_path, "rb") as file:
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=file,
                prompt=prompt_text,
            )
        return transcript.text.strip()

    # Whisper uploads are limited to 25 MiB, so keep a small safety margin.
    if audio_file_size <= max_chunk_bytes:
        return transcribe_file(audio_path)

    # Chunk by duration using a conservative estimate based on the source file.
    chunk_duration_ms = max(60_000, int(len(audio) * (max_chunk_bytes / audio_file_size) * 0.9))
    transcripts = []

    with TemporaryDirectory() as tmp_dir:
        tmp_dir = Path(tmp_dir)
        for index, start_ms in enumerate(range(0, len(audio), chunk_duration_ms), start=1):
            chunk = audio[start_ms : start_ms + chunk_duration_ms]
            chunk_path = tmp_dir / f"chunk_{index:03d}.m4a"
            chunk.export(chunk_path, format="ipod")

            if chunk_path.stat().st_size > max_chunk_bytes:
                raise ValueError(
                    f"Chunk {index} is still too large to upload: {chunk_path.stat().st_size} bytes"
                )

            chunk_text = transcribe_file(chunk_path)
            if chunk_text:
                transcripts.append(chunk_text)

    return "\n".join(transcripts)

# 3. Add any useful words, names, or technical terms
prompt_text = (
    "Talk or interview about trustworthy AI and responsible AI systems. "
    "Topics include AI risk, AI safety, governance, regulations, ethics, "
    "transparency, accountability, fairness, bias, and trust in AI. "
    "May mention large language models, foundation models, compliance, "
    "data protection, and alignment."
)

# 4. Run the transcription with the prompt
prompted_text = transcribe_with_prompt(audio_path, prompt_text)

print("Prompted transcription:")
print("-" * 50)
print(prompted_text)

# 5. Save transcript to /sources as txt
output_filename = "The_Blueprint_For_Trustworthy_AI_transcript.txt"
output_path = audio_dir / output_filename

with open(output_path, "w", encoding="utf-8") as f:
    f.write(prompted_text)

print("Transcription saved to:", output_path)



The audio file is ready.
Audio path: sources\The_Blueprint_For_Trustworthy_AI.m4a
Prompted transcription:
--------------------------------------------------
So imagine for a second you're driving across, I don't know, a massive suspension bridge. Okay. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's there. Exactly. But now let's switch gears. Think about the algorithm that just denied your mortgage application, or the AI system scanning your face at the airport. Do you have that same trust? And I mean, honestly, should you? That is the defining question of our decade, I think. We've moved past the wow phase of AI and straight into the wait a minute phase. Today we are digging into the blue

In [5]:
#stabforming and savign the .pdf file into .txt
from pathlib import Path
from PyPDF2 import PdfReader

sources_dir = Path("sources")
pdf_path = sources_dir / "385082eng.pdf"
txt_path = sources_dir / "385082eng_text.txt"

# Read PDF
reader = PdfReader(pdf_path)
pages_text = []

for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        pages_text.append(page_text)

full_text = "\n\n".join(pages_text)

# Save to txt
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(full_text)

print("Saved PDF text to:", txt_path)
print("Number of characters:", len(full_text))

Saved PDF text to: sources\385082eng_text.txt
Number of characters: 22101


## Step 2: Implement Fixed-Size Chunking

In [6]:
from pathlib import Path

sources_dir = Path("sources")

podcast_txt_path = sources_dir / "The_Blueprint_For_Trustworthy_AI_transcript.txt"
pdf_txt_path = sources_dir / "385082eng_text.txt" 

with open(podcast_txt_path, "r", encoding="utf-8") as f:
    podcast_text = f.read()

with open(pdf_txt_path, "r", encoding="utf-8") as f:
    pdf_text = f.read()

print("Podcast characters:", len(podcast_text))
print("PDF characters:", len(pdf_text))

Podcast characters: 16186
PDF characters: 22101


In [21]:
# Use CharacterTextSplitter with fixed chunk size
# test run at 500

from langchain_text_splitters import CharacterTextSplitter

fixed_splitter_500_0 = CharacterTextSplitter(
    separator="",      # try to split on paragraph breaks when possible
    chunk_size=500,        # target ~500 characters per chunk
    chunk_overlap=0,       # no overlap
    length_function=len,   # measure length by characters
    is_separator_regex=False
)

podcast_chunks_500_0 = fixed_splitter_500_0.split_text(podcast_text)
pdf_chunks_500_0 = fixed_splitter_500_0.split_text(pdf_text)

print("Podcast chunks (500,0):", len(podcast_chunks_500_0))
print("PDF chunks (500,0):", len(pdf_chunks_500_0))

Podcast chunks (500,0): 33
PDF chunks (500,0): 45


In [22]:
# Use CharacterTextSplitter with fixed chunk size
# Loop run with different sizes: 500, 1000, 2000 char. and overlap values: 0, 50, 100 char.

from langchain_text_splitters import CharacterTextSplitter

configs = [
    (500, 0),
    (500, 50),
    (500, 100),
    (1000, 0),
    (1000, 50),
    (2000, 0),
]

fixed_results = []

for chunk_size, chunk_overlap in configs:
    splitter = CharacterTextSplitter(
        separator="",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        is_separator_regex=False,
    )

    podcast_chunks = splitter.split_text(podcast_text)
    pdf_chunks = splitter.split_text(pdf_text)

    avg_podcast_len = sum(len(c) for c in podcast_chunks) / len(podcast_chunks)
    avg_pdf_len = sum(len(c) for c in pdf_chunks) / len(pdf_chunks)

    fixed_results.append({
        "Chunk size": chunk_size,
        "Overlap": chunk_overlap,
        "# Podcast chunks": len(podcast_chunks),
        "Avg podcast len (chars)": avg_podcast_len,
        "# PDF chunks": len(pdf_chunks),
        "Avg PDF len (chars)": avg_pdf_len,
    })

fixed_results

[{'Chunk size': 500,
  'Overlap': 0,
  '# Podcast chunks': 33,
  'Avg podcast len (chars)': 490.09090909090907,
  '# PDF chunks': 45,
  'Avg PDF len (chars)': 490.8666666666667},
 {'Chunk size': 500,
  'Overlap': 50,
  '# Podcast chunks': 36,
  'Avg podcast len (chars)': 497.8333333333333,
  '# PDF chunks': 50,
  'Avg PDF len (chars)': 490.58},
 {'Chunk size': 500,
  'Overlap': 100,
  '# Podcast chunks': 41,
  'Avg podcast len (chars)': 491.9756097560976,
  '# PDF chunks': 56,
  'Avg PDF len (chars)': 492.57142857142856},
 {'Chunk size': 1000,
  'Overlap': 0,
  '# Podcast chunks': 17,
  'Avg podcast len (chars)': 951.6470588235294,
  '# PDF chunks': 23,
  'Avg PDF len (chars)': 960.6521739130435},
 {'Chunk size': 1000,
  'Overlap': 50,
  '# Podcast chunks': 17,
  'Avg podcast len (chars)': 998.8823529411765,
  '# PDF chunks': 24,
  'Avg PDF len (chars)': 968.2916666666666},
 {'Chunk size': 2000,
  'Overlap': 0,
  '# Podcast chunks': 9,
  'Avg podcast len (chars)': 1798.111111111111,
  

In [23]:
# Display as a nicely formatted table with pandas
import pandas as pd

fixed_df = pd.DataFrame(fixed_results)

# Optional: round averages for display
fixed_df["Avg podcast len (chars)"] = fixed_df["Avg podcast len (chars)"].round(1)
fixed_df["Avg PDF len (chars)"] = fixed_df["Avg PDF len (chars)"].round(1)

fixed_df

,Chunk size,Overlap,# Podcast chunks,Avg podcast len (chars),# PDF chunks,Avg PDF len (chars)
0,500,0,33,490.1,45,490.9
1,500,50,36,497.8,50,490.6
2,500,100,41,492.0,56,492.6
3,1000,0,17,951.6,23,960.7
4,1000,50,17,998.9,24,968.3
5,2000,0,9,1798.1,12,1841.5


In [31]:
# Savign the results in a markdown format at reports/

import pandas as pd  # if not already imported
from pathlib import Path

# 1. Generate Markdown from the DataFrame
def fixed_df_to_markdown(df: pd.DataFrame) -> str:
    header = (
        "| Chunk size | Overlap | # Podcast chunks | Avg podcast len (chars) "
        "| # PDF chunks | Avg PDF len (chars) |\n"
    )
    header += (
        "|-----------|---------|------------------|-------------------------"
        "|--------------|----------------------|\n"
    )

    rows = []
    for _, r in df.iterrows():
        row = (
            f"| {int(r['Chunk size'])} "
            f"| {int(r['Overlap'])} "
            f"| {int(r['# Podcast chunks'])} "
            f"| {r['Avg podcast len (chars)']:.1f} "
            f"| {int(r['# PDF chunks'])} "
            f"| {r['Avg PDF len (chars)']:.1f} |"
        )
        rows.append(row)

    return header + "\n".join(rows)

fixed_table_md = fixed_df_to_markdown(fixed_df)
print(f"Preview first ~100 chars:\n{fixed_table_md[:100]}")


# 2. Define the reports directory
reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)  # create if it doesn't exist

# 3. Choose a descriptive filename
md_path = reports_dir / "fixed_size_chunking_podcast_vs_pdf.md"

# 4. Write the markdown content
with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Fixed-Size Chunking Results (Podcast vs PDF)\n\n")
    f.write("This table summarizes the number and average length of chunks for different "
            "fixed-size and overlap settings, comparing podcast transcript and PDF text.\n\n")
    f.write(fixed_table_md)
    f.write("\n")

print(f"\nSaved Markdown table to:", md_path)

Preview first ~100 chars:
| Chunk size | Overlap | # Podcast chunks | Avg podcast len (chars) | # PDF chunks | Avg PDF len (ch

Saved Markdown table to: reports\fixed_size_chunking_podcast_vs_pdf.md


Analysis 

In [32]:
def show_chunk_endings(name, chunks, n=5):
    print(f"\n=== {name} (showing last 100 chars of first {n} chunks) ===")
    for i, ch in enumerate(chunks[:n], start=1):
        print(f"\n--- Chunk {i} ---")
        print(ch[-100:])  # last 100 characters

# Example: inspect 500,0 config we already created
show_chunk_endings("Podcast 500,0", podcast_chunks_500_0, n=5)
show_chunk_endings("PDF 500,0", pdf_chunks_500_0, n=5)


=== Podcast 500,0 (showing last 100 chars of first 5 chunks) ===

--- Chunk 1 ---
 the infrastructure. It's invisible, but it's there. Exactly. But now let's switch gears. Think abou

--- Chunk 2 ---
 arguably the Magna Carta for ethical computing. The Ethics Guidelines for Trustworthy AI. This is a

--- Chunk 3 ---
explosion we have today. So why are we blowing the dust off this specific report? Because it's not d

--- Chunk 4 ---
tract concept like fairness and turn it into, I don't know, Python code. We've got three pillars, fo

--- Chunk 5 ---
k of it like a three-legged stool. The first leg is lawful. The AI has to comply with all the regula

=== PDF 500,0 (showing last 100 chars of first 5 chunks) ===

--- Chunk 1 ---
al-ShareAlike 3.0 IGO (CC-BY-NC-SA 3.0 IGO) license 
(http://creativecommons.org/licenses/by-nc-sa/3

--- Chunk 2 ---
nciples lay out a Human Rights-centred 
Approach to the Ethics of AI·Message from Gabriela Ramos
Ass

--- Chunk 3 ---
 unique mandate, UNESCO’s Soc

In [35]:
# Function to capture chunk endings as a Markdown-like string
def chunk_endings_markdown(name: str, chunks, n: int = 5) -> str:
    lines = []
    lines.append(f"## {name} (last 100 characters of first {n} chunks)\n")
    for i, ch in enumerate(chunks[:n], start=1):
        lines.append(f"### Chunk {i}\n")
        # Use fenced code block for readability in Markdown
        lines.append("```text")
        lines.append(ch[-100:])
        lines.append("```")
        lines.append("")  # blank line
    return "\n".join(lines)

podcast_endings_md = chunk_endings_markdown("Podcast 500,0", podcast_chunks_500_0, n=5)
pdf_endings_md = chunk_endings_markdown("PDF 500,0", pdf_chunks_500_0, n=5)

# Save both inspections into a single .md report

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

md_path = reports_dir / "fixed_chunking_boundary_examples.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Fixed-Size Chunking Boundary Examples\n\n")
    f.write("This file shows the last 100 characters of the first few chunks "
            "for the podcast transcript and the PDF text using the fixed-size "
            "500-character, 0-overlap configuration.\n\n")
    f.write(podcast_endings_md)
    f.write("\n\n")
    f.write(pdf_endings_md)
    f.write("\n")

print("Saved chunk boundary examples to:", md_path)

Saved chunk boundary examples to: reports\fixed_chunking_boundary_examples.md


## Step 3: Implement Recursive Character Chunking

In [36]:
# Basic recursive splitter setup
from langchain_text_splitters import RecursiveCharacterTextSplitter

# test
recursive_splitter_500_0 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=0,
    # Let it use the default separator priority:
    # ["\n\n", "\n", " ", ""]
)

podcast_recursive_500_0 = recursive_splitter_500_0.split_text(podcast_text)
pdf_recursive_500_0 = recursive_splitter_500_0.split_text(pdf_text)

len(podcast_recursive_500_0), len(pdf_recursive_500_0)


(33, 54)

In [37]:
# Experiment with multiple chunk sizes (500, 1000, 2000)
recursive_configs = [
    (500, 0),
    (500, 50),
    (1000, 0),
    (1000, 50),
    (2000, 0),
]

recursive_results = []

for chunk_size, chunk_overlap in recursive_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    podcast_chunks = splitter.split_text(podcast_text)
    pdf_chunks = splitter.split_text(pdf_text)

    avg_podcast_len = sum(len(c) for c in podcast_chunks) / len(podcast_chunks)
    avg_pdf_len = sum(len(c) for c in pdf_chunks) / len(pdf_chunks)

    recursive_results.append({
        "Chunk size": chunk_size,
        "Overlap": chunk_overlap,
        "# Podcast chunks": len(podcast_chunks),
        "Avg podcast len (chars)": avg_podcast_len,
        "# PDF chunks": len(pdf_chunks),
        "Avg PDF len (chars)": avg_pdf_len,
    })

In [38]:
recursive_df = pd.DataFrame(recursive_results)
recursive_df["Avg podcast len (chars)"] = recursive_df["Avg podcast len (chars)"].round(1)
recursive_df["Avg PDF len (chars)"] = recursive_df["Avg PDF len (chars)"].round(1)
recursive_df

,Chunk size,Overlap,# Podcast chunks,Avg podcast len (chars),# PDF chunks,Avg PDF len (chars)
0,500,0,33,489.5,54,407.3
1,500,50,37,481.2,54,425.9
2,1000,0,17,951.2,30,734.7
3,1000,50,17,991.8,30,747.1
4,2000,0,9,1797.6,17,1297.9


In [42]:
# Adjust separator priority (podcast vs PDF)

# Podcast: prioritize sentences and words
podcast_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

podcast_chunks_custom = podcast_splitter.split_text(podcast_text)
print("Podcast chunks (custom separators):", len(podcast_chunks_custom))
print("Podcast sample chunk end:\n", podcast_chunks_custom[0][-50:])

# PDF: prioritize big section and paragraph breaks
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n\n", "\n\n", "\n", ". ", " ", ""]
)

pdf_chunks_custom = pdf_splitter.split_text(pdf_text)
print("\nPDF chunks (custom separators):", len(pdf_chunks_custom))
print("PDF sample chunk end:\n", pdf_chunks_custom[0][-50:])

Podcast chunks (custom separators): 37
Podcast sample chunk end:
 ut it's there. Exactly. But now let's switch gears

PDF chunks (custom separators): 54
PDF sample chunk end:
 e Ethics
 of Artificial
Intelligence
Supported by:


In [47]:
def chunk_endings_markdown(name: str, chunks, n: int = 5) -> str:
    lines = []
    lines.append(f"## {name} (last 100 characters of first {n} chunks)\n")
    for i, ch in enumerate(chunks[:n], start=1):
        lines.append(f"### Chunk {i}\n")
        lines.append("```text")
        lines.append(ch[-100:])
        lines.append("```")
        lines.append("")
    return "\n".join(lines)

podcast_recursive_endings_md = chunk_endings_markdown("Podcast recursive 500,0", podcast_recursive_500_0, n=5)
pdf_recursive_endings_md = chunk_endings_markdown("PDF recursive 500,0", pdf_recursive_500_0, n=5)

In [48]:
from pathlib import Path

reports_dir = Path("reports")
reports_dir.mkdir(exist_ok=True)

md_path = reports_dir / "recursive_chunking_boundary_examples.md"

with open(md_path, "w", encoding="utf-8") as f:
    f.write("# Recursive Chunking Boundary Examples\n\n")
    f.write("Last 100 chars of the first few recursive chunks for podcast and PDF.\n\n")
    f.write(podcast_recursive_endings_md)
    f.write("\n\n")
    f.write(pdf_recursive_endings_md)
    f.write("\n")

print("Saved recursive boundary examples to:", md_path)

Saved recursive boundary examples to: reports\recursive_chunking_boundary_examples.md
